# YouTube Automation — @clarityinthequran

### How to use this
Every cell below is a **form**. You never edit code.
Fill in the boxes, tap the **▶** button on the left. That is it.

1. **Run STEP 1 first**, every session. (~40 seconds)
2. Then tap whichever action you need.

If STEP 1 complains about something, it will tell you exactly which
Cloud Console page to fix and give you the link.

In [ ]:
#@title ▶️ STEP 1 — Connect me to YouTube (run this first) { display-mode: "form" }
#@markdown Leave these alone unless you want different folder names.
#@markdown Then tap the **▶** button to the left.
#@markdown
UPLOAD_FOLDER_NAME = "YouTube Uploads" #@param {type:"string"}
BACKUP_FOLDER_NAME = "YouTube Backups" #@param {type:"string"}
SECRETS_FOLDER_NAME = "YouTube Secrets" #@param {type:"string"}
#@markdown ---

import os
import sys
import glob
import json
import time
import random
import shutil
import subprocess

print("Installing... (about 40 seconds, only on first run)")

# yt-dlp MUST be the newest build or YouTube breaks it. The
# Google libraries ship with Colab already, so we only fill gaps
# rather than upgrading them mid-session and risking a restart.
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "--upgrade",
     "yt-dlp"], check=False)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q",
     "google-api-python-client", "google-auth-oauthlib",
     "google-auth-httplib2"], check=False)

from google.colab import drive
from google.colab import auth

if not os.path.ismount("/content/drive"):
    drive.mount("/content/drive")
try:
    auth.authenticate_user()
except Exception:
    pass  # not fatal; YouTube auth is separate

DRIVE_ROOT = "/content/drive/MyDrive"
UPLOAD_FOLDER_PATH = DRIVE_ROOT + "/" + UPLOAD_FOLDER_NAME
BACKUP_FOLDER_PATH = DRIVE_ROOT + "/" + BACKUP_FOLDER_NAME
SECRETS_FOLDER_PATH = DRIVE_ROOT + "/" + SECRETS_FOLDER_NAME

# I make the folders for you -- you never touch the Drive app.
for FOLDER in (UPLOAD_FOLDER_PATH, BACKUP_FOLDER_PATH,
               SECRETS_FOLDER_PATH):
    os.makedirs(FOLDER, exist_ok=True)

CLIENT_SECRET_FILE = SECRETS_FOLDER_PATH + "/client_secret.json"
TOKEN_FILE = SECRETS_FOLDER_PATH + "/token.json"


def find_client_secret():
    """
    Locate the JSON Google made you download, wherever it landed.
    You do NOT have to rename it or move it -- I do that for you.
    """
    if os.path.exists(CLIENT_SECRET_FILE):
        return CLIENT_SECRET_FILE

    # Search shallow first (fast), then everywhere (slower).
    PATTERNS = [
        SECRETS_FOLDER_PATH + "/client_secret*.json",
        DRIVE_ROOT + "/client_secret*.json",
        DRIVE_ROOT + "/*/client_secret*.json",
        DRIVE_ROOT + "/*/*/client_secret*.json",
    ]
    HITS = []
    for PATTERN in PATTERNS:
        HITS = sorted(glob.glob(PATTERN))
        if HITS:
            break

    if not HITS:
        print("Searching your whole Drive, one moment...")
        HITS = sorted(glob.glob(
            DRIVE_ROOT + "/**/client_secret*.json", recursive=True))

    if not HITS:
        return None

    # Tidy it away so every future run is instant.
    shutil.copy(HITS[0], CLIENT_SECRET_FILE)
    print("Found your credentials file and filed it away.")
    return CLIENT_SECRET_FILE


SETUP_HELP = """
--------------------------------------------------------------
 I could not find your Google credentials file.
 That is the one thing I cannot create for you -- it has to be
 made while signed into your own Google account.

 Six taps, once. Tap 'aA' in Safari's address bar first and
 choose 'Request Desktop Website', then:

  1. console.cloud.google.com
     -> project dropdown -> New Project -> Create -> select it
  2. console.cloud.google.com/apis/library/youtube.googleapis.com
     -> Enable
  3. console.cloud.google.com/auth/overview
     -> Get started -> your name/email -> External -> Create
  4. console.cloud.google.com/auth/scopes
     -> Add or remove scopes -> filter 'youtube'
     -> tick youtube.upload AND youtube.force-ssl -> Update -> Save
  5. console.cloud.google.com/auth/clients
     -> Create client -> type MUST be 'Desktop app' -> Create
     -> Download JSON
  6. console.cloud.google.com/auth/audience
     -> Publish app   (skip this and your login dies in 7 days)

 Then just save that downloaded file anywhere in your Drive and
 run this cell again. No renaming, no moving -- I will find it.
--------------------------------------------------------------
"""

CLIENT_SECRET_FILE_FOUND = find_client_secret()
if not CLIENT_SECRET_FILE_FOUND:
    print(SETUP_HELP)
    raise SystemExit("Waiting on your credentials file.")

from urllib.parse import urlparse, parse_qs
from google_auth_oauthlib.flow import Flow
from google.oauth2.credentials import Credentials
from google.auth.transport.requests import Request
from googleapiclient.discovery import build
from googleapiclient.http import MediaFileUpload
from googleapiclient.errors import HttpError

YOUTUBE_SCOPES = [
    "https://www.googleapis.com/auth/youtube.upload",
    "https://www.googleapis.com/auth/youtube.force-ssl",
]

# Google blocked the old paste-a-code flow in 2023, so we bounce
# off localhost. Your phone cannot open localhost -- the page will
# fail, and that failure is exactly what we want.
REDIRECT_URI = "http://localhost:8080/"
os.environ["OAUTHLIB_INSECURE_TRANSPORT"] = "1"
os.environ["OAUTHLIB_RELAX_TOKEN_SCOPE"] = "1"


def connect_to_youtube():
    """Return a ready YouTube client, asking you for as little
    as humanly possible."""
    CREDS = None

    if os.path.exists(TOKEN_FILE):
        try:
            CREDS = Credentials.from_authorized_user_file(
                TOKEN_FILE, YOUTUBE_SCOPES)
        except Exception:
            CREDS = None

    if CREDS and CREDS.expired and CREDS.refresh_token:
        try:
            CREDS.refresh(Request())
        except Exception:
            # Usually the 7-day expiry from step 6 being skipped.
            print("Your saved login expired. Signing in again.")
            os.remove(TOKEN_FILE)
            CREDS = None

    if not CREDS or not CREDS.valid:
        FLOW = Flow.from_client_secrets_file(
            CLIENT_SECRET_FILE_FOUND,
            scopes=YOUTUBE_SCOPES,
            redirect_uri=REDIRECT_URI,
        )
        AUTH_URL, _ = FLOW.authorization_url(
            access_type="offline",
            prompt="consent",
            include_granted_scopes="true",
        )
        print("")
        print("=" * 56)
        print("  ONE-TIME SIGN IN -- about 30 seconds")
        print("=" * 56)
        print("")
        print("  1. Long-press this link, Open in New Tab:")
        print("")
        print(AUTH_URL)
        print("")
        print("  2. Choose your channel's Google account.")
        print("  3. 'Google hasn't verified this app' ->")
        print("     tap Advanced -> Go to ... (unsafe).")
        print("     It is your own app. This is safe.")
        print("  4. Tap Continue to approve both permissions.")
        print("  5. Safari says 'cannot connect to the server'.")
        print("     >>> THAT MEANS IT WORKED. <<<")
        print("  6. Tap the address bar, Select All, Copy.")
        print("  7. Come back here and paste it below.")
        print("")
        PASTED = input("  Paste here, then press enter: ").strip()

        if PASTED.startswith("http"):
            CODE = parse_qs(urlparse(PASTED).query).get("code", [""])[0]
        else:
            CODE = PASTED
        if not CODE:
            raise RuntimeError(
                "That did not contain a code. Copy the URL from the "
                "page that FAILED to load, not the sign-in page.")

        FLOW.fetch_token(code=CODE)
        CREDS = FLOW.credentials
        with open(TOKEN_FILE, "w") as HANDLE:
            HANDLE.write(CREDS.to_json())
        print("")
        print("Signed in. You will not have to do that again.")

    return build("youtube", "v3", credentials=CREDS)


try:
    YOUTUBE = connect_to_youtube()
except Exception as ERROR:
    TEXT = str(ERROR)
    if "access_denied" in TEXT:
        print("\nGoogle refused the sign-in. Go to")
        print("console.cloud.google.com/auth/audience")
        print("and tap 'Publish app', then run this cell again.")
    elif "redirect_uri_mismatch" in TEXT:
        print("\nYour credential is the wrong type. Go to")
        print("console.cloud.google.com/auth/clients")
        print("and make a new client of type 'Desktop app'.")
    elif "invalid_scope" in TEXT or "insufficient" in TEXT:
        print("\nMissing permissions. Go to")
        print("console.cloud.google.com/auth/scopes")
        print("and tick youtube.upload AND youtube.force-ssl.")
    raise

ME = YOUTUBE.channels().list(
    part="snippet,contentDetails,statistics", mine=True).execute()
MY_CHANNEL = ME["items"][0]
MY_CHANNEL_ID = MY_CHANNEL["id"]
MY_UPLOADS_PLAYLIST = (
    MY_CHANNEL["contentDetails"]["relatedPlaylists"]["uploads"])

VIDEO_TYPES = (".mp4", ".mov", ".m4v", ".webm", ".mkv", ".avi")
CATEGORY_IDS = {
    "Education": "27",
    "People & Blogs": "22",
    "Entertainment": "24",
    "News & Politics": "25",
    "Nonprofits & Activism": "29",
}
LAST_COMMENT_LIST = []


def videos_waiting():
    """Video files sitting in your upload folder, newest first."""
    FOUND = [N for N in os.listdir(UPLOAD_FOLDER_PATH)
             if N.lower().endswith(VIDEO_TYPES)]
    FOUND.sort(
        key=lambda N: os.path.getmtime(
            os.path.join(UPLOAD_FOLDER_PATH, N)),
        reverse=True,
    )
    return FOUND


print("")
print("=" * 56)
print("  READY.")
print("=" * 56)
print("  Channel      :", MY_CHANNEL["snippet"]["title"])
print("  Videos live  :", MY_CHANNEL["statistics"].get("videoCount"))
WAITING = videos_waiting()
print("  Waiting here :", len(WAITING), "file(s) in",
      UPLOAD_FOLDER_NAME)
for NAME in WAITING[:5]:
    print("     -", NAME)
print("")
print("  Drop CapCut exports into the '" + UPLOAD_FOLDER_NAME + "'")
print("  folder in Drive, then use the UPLOAD cell below.")
print("=" * 56)

In [ ]:
#@title ⬆️ UPLOAD A VIDEO { display-mode: "form" }
#@markdown Type a title, tap **▶**. By default it grabs the
#@markdown **newest** video in your uploads folder — so you can
#@markdown export from CapCut and come straight here.
#@markdown
VIDEO_TITLE = "" #@param {type:"string"}
DESCRIPTION = "" #@param {type:"string"}
TAGS_COMMA_SEPARATED = "quran, tafsir, islam" #@param {type:"string"}
CATEGORY = "Education" #@param ["Education", "People & Blogs", "Entertainment", "News & Politics", "Nonprofits & Activism"]
PRIVACY = "private" #@param ["private", "unlisted", "public"]
#@markdown ---
#@markdown Untick only if you want a different file:
USE_NEWEST_VIDEO = True #@param {type:"boolean"}
OR_TYPE_A_FILENAME = "" #@param {type:"string"}
#@markdown ---

WAITING = videos_waiting()

if USE_NEWEST_VIDEO:
    if not WAITING:
        raise SystemExit(
            "No videos found in '" + UPLOAD_FOLDER_NAME + "'. "
            "Export from CapCut to that Drive folder first.")
    CHOSEN_FILE = WAITING[0]
else:
    CHOSEN_FILE = OR_TYPE_A_FILENAME.strip()
    if not CHOSEN_FILE:
        raise SystemExit("Type a filename, or tick USE_NEWEST_VIDEO.")

if not VIDEO_TITLE.strip():
    raise SystemExit("Give it a title first.")

SOURCE_PATH = os.path.join(UPLOAD_FOLDER_PATH, CHOSEN_FILE)
if not os.path.exists(SOURCE_PATH):
    print("Cannot find that file. These are available:")
    for NAME in WAITING:
        print("   -", NAME)
    raise SystemExit("File not found: " + CHOSEN_FILE)

print("Uploading :", CHOSEN_FILE)

# Drive's mount stalls on long uploads, so copy to local disk.
LOCAL_PATH = "/content/" + CHOSEN_FILE
shutil.copy(SOURCE_PATH, LOCAL_PATH)

TAG_LIST = [T.strip() for T in TAGS_COMMA_SEPARATED.split(",")
            if T.strip()]

REQUEST = YOUTUBE.videos().insert(
    part="snippet,status",
    body={
        "snippet": {
            "title": VIDEO_TITLE.strip(),
            "description": DESCRIPTION.strip(),
            "tags": TAG_LIST,
            "categoryId": CATEGORY_IDS[CATEGORY],
        },
        "status": {
            "privacyStatus": PRIVACY,
            "selfDeclaredMadeForKids": False,
        },
    },
    media_body=MediaFileUpload(LOCAL_PATH,
                               chunksize=5 * 1024 * 1024,
                               resumable=True,
                               mimetype="video/*"),
)

RESPONSE = None
RETRIES = 0
while RESPONSE is None:
    try:
        STATUS, RESPONSE = REQUEST.next_chunk()
        if STATUS:
            print("   %d%%" % int(STATUS.progress() * 100))
    except HttpError as ERROR:
        TRANSIENT = (429, 500, 502, 503, 504)
        if ERROR.resp.status in TRANSIENT and RETRIES < 5:
            RETRIES += 1
            time.sleep((2 ** RETRIES) + random.random())
        elif ERROR.resp.status == 403 and "quota" in str(ERROR):
            raise SystemExit(
                "Daily quota spent (6 uploads/day). "
                "Resets at midnight Pacific.")
        else:
            raise

os.remove(LOCAL_PATH)
VIDEO_ID = RESPONSE["id"]

print("")
print("=" * 56)
print("  DONE")
print("=" * 56)
print("  https://youtu.be/" + VIDEO_ID)
print("")
print("  Heads up: YouTube locks API uploads to PRIVATE until")
print("  your project passes their audit. To publish it, open")
print("  the YouTube Studio app and switch it to Public.")
print("=" * 56)

In [ ]:
#@title 💬 SHOW ME MY COMMENTS { display-mode: "form" }
#@markdown Tap **▶**. Each comment gets a number — use that
#@markdown number in the REPLY cell below.
#@markdown
SHOW = "only ones I haven't replied to" #@param ["only ones I haven't replied to", "all recent comments"]
HOW_MANY_TO_CHECK = 25 #@param {type:"slider", min:5, max:100, step:5}
#@markdown ---

COLLECTED = []
PAGE_TOKEN = None

while len(COLLECTED) < HOW_MANY_TO_CHECK:
    RESULT = YOUTUBE.commentThreads().list(
        part="snippet,replies",
        allThreadsRelatedToChannelId=MY_CHANNEL_ID,
        maxResults=min(100, HOW_MANY_TO_CHECK - len(COLLECTED)),
        order="time",
        textFormat="plainText",
        pageToken=PAGE_TOKEN,
    ).execute()

    for THREAD in RESULT.get("items", []):
        TOP = THREAD["snippet"]["topLevelComment"]
        SNIP = TOP["snippet"]
        COLLECTED.append({
            "comment_id": TOP["id"],
            "video_id": SNIP.get("videoId"),
            "author": SNIP["authorDisplayName"],
            "text": SNIP["textDisplay"],
            "likes": SNIP["likeCount"],
            "reply_count": THREAD["snippet"]["totalReplyCount"],
        })

    PAGE_TOKEN = RESULT.get("nextPageToken")
    if not PAGE_TOKEN:
        break

if SHOW.startswith("only"):
    SHOWN = [C for C in COLLECTED if C["reply_count"] == 0]
else:
    SHOWN = COLLECTED

# Remember these so the REPLY cell can use plain numbers.
LAST_COMMENT_LIST = SHOWN

if not SHOWN:
    print("Nothing to show. All caught up.")
else:
    for INDEX, ITEM in enumerate(SHOWN, 1):
        print("=" * 52)
        print(" #%d   %s   (%d likes)" %
              (INDEX, ITEM["author"], ITEM["likes"]))
        print("")
        print(" " + ITEM["text"].replace("\n", "\n "))
    print("=" * 52)
    print("")
    print(len(SHOWN), "comment(s). To answer #1, use the")
    print("REPLY cell below and set COMMENT_NUMBER to 1.")

In [ ]:
#@title ↩️ REPLY TO A COMMENT { display-mode: "form" }
#@markdown Run the comments cell above first, then put its
#@markdown number here and type your reply. Tap **▶**.
#@markdown
COMMENT_NUMBER = 1 #@param {type:"integer"}
YOUR_REPLY = "" #@param {type:"string"}
#@markdown ---

if not LAST_COMMENT_LIST:
    raise SystemExit(
        "Run the '💬 SHOW ME MY COMMENTS' cell first.")

if not YOUR_REPLY.strip():
    raise SystemExit("Type a reply first.")

if not 1 <= COMMENT_NUMBER <= len(LAST_COMMENT_LIST):
    raise SystemExit(
        "Pick a number between 1 and %d."
        % len(LAST_COMMENT_LIST))

TARGET = LAST_COMMENT_LIST[COMMENT_NUMBER - 1]

YOUTUBE.comments().insert(
    part="snippet",
    body={"snippet": {"parentId": TARGET["comment_id"],
                      "textOriginal": YOUR_REPLY.strip()}},
).execute()

print("Replied to", TARGET["author"])
print("")
print("  they said : " + TARGET["text"][:80])
print("  you said  : " + YOUR_REPLY.strip())

In [ ]:
#@title 💾 BACK UP MY CHANNEL TO DRIVE { display-mode: "form" }
#@markdown Downloads your videos into your backups folder.
#@markdown Safe to run again and again — it only fetches what
#@markdown it has not already saved. Tap **▶**.
#@markdown
HOW_MANY_THIS_RUN = 5 #@param {type:"slider", min:1, max:50, step:1}
QUALITY = "1080p" #@param ["1080p", "4K if available", "720p (small)"]
#@markdown ---

from yt_dlp import YoutubeDL

QUALITY_MAP = {
    "1080p": "bv*[height<=1080]+ba/b[height<=1080]/b",
    "4K if available": "bv*+ba/b",
    "720p (small)": "bv*[height<=720]+ba/b[height<=720]/b",
}

ARCHIVE_FILE = BACKUP_FOLDER_PATH + "/_already_backed_up.txt"
LOCAL_WORK_DIR = "/content/yt_backup_tmp"
COOKIES_FILE_PATH = SECRETS_FOLDER_PATH + "/cookies.txt"
os.makedirs(LOCAL_WORK_DIR, exist_ok=True)

# The uploads playlist includes Shorts; the /videos tab does not.
CHANNEL_SOURCE_URL = ("https://www.youtube.com/playlist?list="
                      + MY_UPLOADS_PLAYLIST)


def download_options():
    OPTIONS = {
        "format": QUALITY_MAP[QUALITY],
        "merge_output_format": "mp4",
        "outtmpl": (LOCAL_WORK_DIR
                    + "/%(upload_date)s_%(title).80B [%(id)s].%(ext)s"),
        "writeinfojson": True,
        "writethumbnail": True,
        "writesubtitles": True,
        "subtitleslangs": ["en.*", "ar.*"],
        "ignoreerrors": True,
        "retries": 5,
        "fragment_retries": 10,
        "concurrent_fragment_downloads": 4,
        "quiet": True,
        "no_warnings": True,
    }
    if os.path.exists(COOKIES_FILE_PATH):
        OPTIONS["cookiefile"] = COOKIES_FILE_PATH
    return OPTIONS


if os.path.exists(ARCHIVE_FILE):
    with open(ARCHIVE_FILE) as HANDLE:
        ALREADY_DONE = {L.strip() for L in HANDLE if L.strip()}
else:
    ALREADY_DONE = set()

FLAT = {"extract_flat": "in_playlist", "ignoreerrors": True,
        "quiet": True, "no_warnings": True}
if os.path.exists(COOKIES_FILE_PATH):
    FLAT["cookiefile"] = COOKIES_FILE_PATH

print("Looking at your channel...")
with YoutubeDL(FLAT) as YDL:
    INFO = YDL.extract_info(CHANNEL_SOURCE_URL, download=False)

ALL_VIDEOS = [{"id": E["id"], "title": E.get("title", "")}
              for E in (INFO.get("entries") or []) if E]
TODO = [V for V in ALL_VIDEOS
        if V["id"] not in ALREADY_DONE][:HOW_MANY_THIS_RUN]

print("  on your channel :", len(ALL_VIDEOS))
print("  already saved   :", len(ALREADY_DONE))
print("  fetching now    :", len(TODO))

if not TODO:
    print("")
    print("Nothing new to back up. You are fully archived.")

FAILURES = 0
for NUMBER, VIDEO in enumerate(TODO, 1):
    print("")
    print("[%d/%d] %s" % (NUMBER, len(TODO), VIDEO["title"][:50]))
    try:
        with YoutubeDL(download_options()) as YDL:
            YDL.download(
                ["https://www.youtube.com/watch?v=" + VIDEO["id"]])
    except Exception as ERROR:
        FAILURES += 1
        print("   skipped:", str(ERROR)[:90])
        continue

    MOVED = 0
    for NAME in os.listdir(LOCAL_WORK_DIR):
        shutil.move(os.path.join(LOCAL_WORK_DIR, NAME),
                    os.path.join(BACKUP_FOLDER_PATH, NAME))
        MOVED += 1
    if MOVED:
        with open(ARCHIVE_FILE, "a") as HANDLE:
            HANDLE.write(VIDEO["id"] + "\n")
        print("   saved to Drive")

print("")
print("Saved into:", BACKUP_FOLDER_NAME)
if FAILURES:
    print("")
    print(FAILURES, "video(s) failed. If it said 'not a bot',")
    print("YouTube is rate-limiting Colab. Try again later, or")
    print("use takeout.google.com for a guaranteed full archive.")